In [1]:
"""
gravitational_field_checker.ipynb
================================
NLP tool that:
  1. Checks whether user input is a grammatically valid sentence.
  2. Checks whether the input discusses the concept of "what is a gravitational field"
     using two complementary methods:
       a) Keyword/phrase matching against the provided corpus
       b) TF-IDF cosine similarity against corpus sentences
"""

import re
import string
import sys
from typing import Optional
from pathlib import Path
import os
import nltk

nltk.data.path.append(str(os.getcwd() + "/pyodide/nltk_data"))

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Corpus ────────────────────────────────────────────────────────────────────

CORPUS_PATH = Path("data/gravitation.txt")
CORPUS_TEXT = CORPUS_PATH.read_text(encoding="utf-8")

# ── Concept definition ────────────────────────────────────────────────────────
# Key phrases that characterise a "what is a gravitational field" discussion.
CONCEPT_KEYWORDS = [
    "gravitational field",
    "gravitational force",
    "gravity",
    "field of gravity",
    "force of gravity",
    "attract",
    "attraction",
    "mass",
    "universal gravitation",
    "newton",
    "what is",
    "definition",
    "describe",
    "explain",
    "region",
    "influence",
    "exerts",
    "surrounding",
]

# Phrases that strongly anchor the "what is" intent
INTENT_PHRASES = [
    r"\bwhat\s+is\b",
    r"\bwhat\s+are\b",
    r"\bdefine\b",
    r"\bdescribe\b",
    r"\bexplain\b",
    r"\bmean(s|ing)?\b",
    r"\bdefine\b",
    r"\bits\b",
    r"\bit is\b",
    r"\bit's\b",
]

# Similarity threshold (0–1) against corpus sentences
SIMILARITY_THRESHOLD = 0.15

STOP_WORDS = set(stopwords.words("english"))


# ── 1. Sentence validity check ────────────────────────────────────────────────

def is_valid_sentence(text: str) -> tuple[bool, str]:
    """
    Return (True, reason) if text looks like a proper sentence, else (False, reason).

    Heuristics used (NLTK-based):
      • Must contain at least one token after stripping punctuation/whitespace
      • Must have a subject-like noun/pronoun AND a verb in POS tags
      • Must not be pure numbers / symbols
    """
    text = text.strip()

    if not text:
        return False, "Input is empty."

    if len(text.split()) < 2:
        return False, "Input is too short to be a sentence (fewer than 2 words)."

    tokens = word_tokenize(text)
    tags = pos_tag(tokens)

    # POS tag sets
    verb_tags   = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "MD"}
    subject_tags = {"NN", "NNS", "NNP", "NNPS", "PRP", "WP", "EX"}

    has_verb    = any(tag in verb_tags    for _, tag in tags)
    has_subject = any(tag in subject_tags for _, tag in tags)

    if not has_verb:
        return False, "No verb detected — input does not appear to be a complete sentence."
    if not has_subject:
        return False, "No noun/subject detected — input does not appear to be a complete sentence."

    return True, "Input is a valid sentence."


# ── 2. Concept relevance check ────────────────────────────────────────────────

def _clean(text: str) -> str:
    """Lowercase and strip punctuation for keyword matching."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text


def keyword_score(text: str) -> tuple[float, list[str]]:
    """
    Return (score 0–1, matched keywords).
    Score = matched unique keywords / total concept keywords.
    """
    cleaned = _clean(text)
    matched = [kw for kw in CONCEPT_KEYWORDS if kw in cleaned]
    score = len(set(matched)) / len(CONCEPT_KEYWORDS)
    return score, list(set(matched))


def has_what_is_intent(text: str) -> bool:
    """True if the input expresses a definitional / explanatory intent."""
    cleaned = text.lower()
    return any(re.search(pat, cleaned) for pat in INTENT_PHRASES)


def tfidf_similarity(user_text: str, corpus: str) -> tuple[float, str]:
    """
    Compute max cosine similarity between user_text and each sentence in the
    corpus.  Returns (max_similarity, most_similar_corpus_sentence).
    """
    corpus_sentences = sent_tokenize(corpus)
    # Filter out very short fragments (figure captions, labels, etc.)
    corpus_sentences = [s for s in corpus_sentences if len(s.split()) > 5]

    docs = corpus_sentences + [user_text]
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    try:
        tfidf_matrix = vectorizer.fit_transform(docs)
    except ValueError:
        return 0.0, ""

    user_vec = tfidf_matrix[-1]          # last row = user input
    corpus_matrix = tfidf_matrix[:-1]    # all but last = corpus

    sims = cosine_similarity(user_vec, corpus_matrix).flatten()
    best_idx = int(np.argmax(sims))
    return float(sims[best_idx]), corpus_sentences[best_idx]


def discusses_gravitational_field(text: str) -> dict:
    """
    Combine keyword and TF-IDF evidence to decide whether the input discusses
    the concept of "what is a gravitational field".

    Returns a result dict with all details.
    """
    kw_score, matched_kws = keyword_score(text)
    sim_score, best_match = tfidf_similarity(text, CORPUS_TEXT)
    intent = has_what_is_intent(text)

    # Decision logic:
    #   • Strong if good similarity AND keywords present
    #   • Moderate if either signal is present along with intent phrasing
    #   • Weak / no if neither signal fires

    on_topic = False
    confidence = "low"

    if kw_score >= 0.15 and sim_score >= SIMILARITY_THRESHOLD:
        on_topic = True
        confidence = "high"
    elif (kw_score >= 0.05 or sim_score >= SIMILARITY_THRESHOLD) and intent:
        on_topic = True
        confidence = "medium-low"
    elif kw_score >= 0.15 or sim_score >= 0.30:
        on_topic = True
        confidence = "medium"

    return {
        "on_topic": on_topic,
        "confidence": confidence,
        "keyword_score": round(kw_score, 3),
        "matched_keywords": matched_kws,
        "tfidf_similarity": round(sim_score, 3),
        "best_corpus_match": best_match.strip(),
        "has_definitional_intent": intent,
    }

STATUS_SUCCESS = 0
STATUS_NOT_SENTENCE = 1
STATUS_OFF_TOPIC = 2

def analyse(user_input: str, show: Optional[bool] = True) -> int:
    """
    Return codes:
    0: Success
    1: Not a full sentence
    2: Off topic
    """
    if show:
        print("\n" + "═" * 60)
        print(f"  Input: {user_input!r}")
        print("═" * 60)

    # Step 1 — sentence check
    valid, sentence_reason = is_valid_sentence(user_input)
    if show:
        print(f"\n[1] Sentence check")
        print(f"    Valid sentence : {'✓ YES' if valid else '✗ NO'}")
        print(f"    Reason         : {sentence_reason}")

    if not valid:
        if show:
            print("\n    ⚠  Stopping here — input is not a valid sentence.")
            print("═" * 60)
        return STATUS_NOT_SENTENCE

    # Step 2 — concept check
    result = discusses_gravitational_field(user_input)
    if show:
        print(f"\n[2] Concept check — 'What is a gravitational field?'")
        print(f"    On topic            : {'✓ YES' if result['on_topic'] else '✗ NO'}")
        print(f"    Confidence          : {result['confidence'].upper()}")
        print(f"    Definitional intent : {'yes' if result['has_definitional_intent'] else 'no'}")
        print(f"    Keyword score       : {result['keyword_score']} "
              f"  (matched: {result['matched_keywords'] or 'none'})")
        print(f"    TF-IDF similarity   : {result['tfidf_similarity']}")
        print(f"    Closest corpus line : \"{result['best_corpus_match'][:120]}…\""
              if len(result['best_corpus_match']) > 120
              else f"    Closest corpus line : \"{result['best_corpus_match']}\"")
        print("═" * 60)
    return STATUS_SUCCESS if result['on_topic'] else STATUS_OFF_TOPIC

def ux(user_input: str):
    """
    What the students actually use and see.
    """
    on_topic_status = analyse(user_input, False)
    if on_topic_status == STATUS_SUCCESS:
        print("Thank you for sharing your thoughts about gravitational fields!")
    elif on_topic_status == STATUS_OFF_TOPIC:
        print("Are you sure this sentence answers the question? If so, move on.")
    else:
        print("This doesn't look like a full sentence yet.")
    
# test cases 
TEST_INPUTS = [
    # Valid + on-topic
    "What is a gravitational field and how does it affect nearby objects?",
    "Can you explain what the gravitational field of a massive object means?",
    "A gravitational field is the region around a mass where gravitational force is exerted.",
    # Valid + off-topic
    "I really enjoy eating pizza on Friday evenings.",
    "The stock market crashed yesterday due to inflation fears.",
    # Valid + partial topic (gravity but no 'what is')
    "Newton described the force of gravity as depending on mass and distance.",
    # Not a sentence
    "gravitational field definition",
    "What",
    "",
]

# Run built-in demo tests
print("\n🔬  Running demo test cases…")
for t in TEST_INPUTS:
    analyse(t)


🔬  Running demo test cases…

════════════════════════════════════════════════════════════
  Input: 'What is a gravitational field and how does it affect nearby objects?'
════════════════════════════════════════════════════════════

[1] Sentence check
    Valid sentence : ✓ YES
    Reason         : Input is a valid sentence.

[2] Concept check — 'What is a gravitational field?'
    On topic            : ✓ YES
    Confidence          : MEDIUM-LOW
    Definitional intent : yes
    Keyword score       : 0.111   (matched: ['what is', 'gravitational field'])
    TF-IDF similarity   : 0.209
    Closest corpus line : "The Moon is not affected by the gravitational field of the Earth."
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  Input: 'Can you explain what the gravitational field of a massive object means?'
════════════════════════════════════════════════════════════

[1] Sentence check
    Valid sentence : ✓ YES


In [2]:
ux("It's a way to visualize how much gravitational force someone would feel at any point.")

Thank you for sharing your thoughts about gravitational fields!


In [3]:
ux("gravitational force everywhere how much")

This doesn't look like a full sentence yet.


In [4]:
ux("the stuff do the thing with the gravity")

Are you sure this sentence is on topic? If so, move on.
